In [17]:
# Vector addition with pycuda(GPU) and numba(CPU), optimized in/out.
# Import and initialize pycuda, numpy, numba, and time
import pycuda.driver as cuda
import pycuda.autoinit
from pycuda.compiler import SourceModule
import numpy as np
import time 
from functools import reduce
from operator import add
from numba import njit, prange

In [18]:

#################### Array size
N = 1024

In [19]:
#################### Create some space on CPU/HOST (random 32-bit ints)
a_cpu = np.random.uniform(1.0, 100.0, size=(N)).astype(np.uint32) 
b_cpu = a_cpu.copy()

In [20]:
#################### Write a GPU kernel
module = SourceModule(""" 
	__global__ void reduction(int* a_gpu){
		// Thread indices
		int tid = threadIdx.x;
		
		for(int l=1; l<blockDim.x; l*=2){
		
			if(tid%(2*l) == 0){
				a_gpu[tid] += a_gpu[tid+l];
			}
			
			__syncthreads();
		}
		
		
	}

""")


In [21]:
total = 0

#################### Start GPU timing
start_gpu = cuda.Event()
end_gpu = cuda.Event()
start_gpu.record()

#################### Launch the GPU kernel
func = module.get_function("reduction")
block_size = 1024
grid_size = int(np.ceil(N/block_size))
func(cuda.InOut(a_cpu), grid=(grid_size, 1, 1), block=(block_size, 1, 1))

#################### End GPU timing
end_gpu.record()
cuda.Context.synchronize()
gpu_time = start_gpu.time_till(end_gpu)*1e-3
print("Elapsed time using GPU (sec): ", gpu_time)
print("total: ", a_cpu[0])
print("---------------------")


Elapsed time using GPU (sec):  0.0007173759937286377
total:  51093
---------------------


In [22]:
#################### Numba CPU parallel reduction

@njit(parallel=True)
def reduce_parallel(a):
    t = 0
    # prange automatically distributes iterations among threads
    for i in prange(N):
        t += a[i]
    return t


In [25]:
#################### Launch Numba CPU version
total = 0
start_cpu = time.time()
total = reduce_parallel(b_cpu)
end_cpu = time.time()
cpu_time = end_cpu - start_cpu
print("Elapsed time using numba CPU(sec): ", cpu_time)
print("total: ", total)
print("---------------------")

Elapsed time using numba CPU(sec):  6.222724914550781e-05
total:  51093
---------------------


In [24]:
#################### Reduce operator 
start_cpu_op = time.time()
total_op = reduce(add, b_cpu)
end_cpu_op = time.time()
cpu_time_op = end_cpu_op - start_cpu_op
print("Elapsed time using sequential reduce function (sec): ", cpu_time_op)
print("total: ", total_op)
print("---------------------")

Elapsed time using sequential reduce function (sec):  0.0001678466796875
total:  51093
---------------------
